# Занятие 1. Реальный мини-агент языковой лаборатории

**Цель практики:** собрать не теоретический пример, а маленького агента, который делает полезную работу для проекта сохранения малоресурсного языка.

Сценарий: у нас есть задача первичной разведки по удмуртскому языку. Агент должен:

1. найти реальные открытые материалы в Wikimedia Commons;
2. выбрать изображение, которое можно скачать;
3. прогнать OCR baseline в бесплатном Colab;
4. взять небольшой контрольный корпус из Удмуртской Википедии;
5. собрать отчет: что найдено, насколько читаемый OCR, что нужно проверить человеку.

Используем `LangGraph`, потому что он хорошо показывает агентское состояние (`state`) и переходы между шагами. LLM/API-ключи не нужны.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-rus
!pip -q install langgraph pytesseract pillow pandas matplotlib requests

In [ ]:
import os, re, json, textwrap, math, statistics, random, io
from pathlib import Path
import pandas as pd
import numpy as np
import requests

DATA_DIR = Path('/content/lowres_lab')
DATA_DIR.mkdir(exist_ok=True)

def show_df(df, n=10):
    display(df.head(n))

def save_artifact(name, obj):
    path = DATA_DIR / name
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    else:
        path.write_text(str(obj), encoding='utf-8')
    print('saved:', path)

from typing import Any, Dict, List, TypedDict
from urllib.parse import quote
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt
import pytesseract
from langgraph.graph import StateGraph, END

## 1. Настройка реальных источников

Берем два открытых источника:

- Wikimedia Commons: категория `Udmurt Dunne`, где лежат реальные файлы, связанные с удмуртской газетой.
- Удмуртская Википедия: небольшой корпус страниц для сравнения с OCR-выводом.

Это не учебная таблица: агент будет обращаться к API и сохранять полученные данные.

In [ ]:
COMMONS_API = 'https://commons.wikimedia.org/w/api.php'
UDM_WIKI_API = 'https://udm.wikipedia.org/w/api.php'

COMMONS_CATEGORY = 'Category:Udmurt Dunne'
LANGUAGE = 'удмуртский'
FALLBACK_FILE_TITLE = 'File:Удномер.jpg'

def api_get(url, params, timeout=30):
    r = requests.get(url, params=params, timeout=timeout, headers={'User-Agent': 'lowres-course-colab/1.0'})
    r.raise_for_status()
    return r.json()

def commons_category_files(category, limit=20):
    data = api_get(COMMONS_API, {
        'action': 'query',
        'list': 'categorymembers',
        'cmtitle': category,
        'cmtype': 'file',
        'cmlimit': limit,
        'format': 'json',
    })
    return data.get('query', {}).get('categorymembers', [])

def commons_imageinfo(title):
    data = api_get(COMMONS_API, {
        'action': 'query',
        'titles': title,
        'prop': 'imageinfo',
        'iiprop': 'url|mime|size|extmetadata',
        'format': 'json',
    })
    pages = data.get('query', {}).get('pages', {})
    page = next(iter(pages.values()))
    info = page.get('imageinfo', [{}])[0]
    meta = info.get('extmetadata', {})
    return {
        'title': title,
        'url': info.get('url'),
        'mime': info.get('mime'),
        'width': info.get('width'),
        'height': info.get('height'),
        'license': meta.get('LicenseShortName', {}).get('value'),
        'artist': meta.get('Artist', {}).get('value'),
        'description': meta.get('ImageDescription', {}).get('value'),
    }

def fetch_udm_wiki_pages(limit=5):
    random_pages = api_get(UDM_WIKI_API, {
        'action': 'query',
        'generator': 'random',
        'grnnamespace': 0,
        'grnlimit': limit,
        'prop': 'extracts',
        'explaintext': 1,
        'exintro': 1,
        'format': 'json',
    })
    pages = []
    for page in random_pages.get('query', {}).get('pages', {}).values():
        title = page.get('title', '')
        extract = page.get('extract', '') or ''
        pages.append({
            'title': title,
            'chars': len(extract),
            'tokens': len(re.findall(r'\w+', extract.lower())),
            'cyrillic_share': cyrillic_share(extract),
            'url': 'https://udm.wikipedia.org/wiki/' + quote(title.replace(' ', '_')),
            'extract_preview': extract[:400],
        })
    return pages

def cyrillic_share(text):
    letters = re.findall(r'[A-Za-zА-Яа-яЁёӜӝӞӟӤӥӦӧӴӵӸӹІіЇїЄєЎўҐґ]', text)
    if not letters:
        return 0.0
    cyr = [x for x in letters if re.match(r'[А-Яа-яЁёӜӝӞӟӤӥӦӧӴӵӸӹІіЇїЄєЎўҐґ]', x)]
    return round(len(cyr) / len(letters), 3)

def text_diagnostics(text):
    tokens = re.findall(r'\w+', text.lower())
    short_tokens = [t for t in tokens if len(t) <= 2]
    weird = re.findall(r'[^\w\s.,:;!?()\-«»"\'/А-Яа-яЁёӜӝӞӟӤӥӦӧӴӵӸӹІіЇїЄєЎўҐґ]', text)
    return {
        'chars': len(text),
        'tokens': len(tokens),
        'cyrillic_share': cyrillic_share(text),
        'short_token_share': round(len(short_tokens) / max(1, len(tokens)), 3),
        'weird_char_count': len(weird),
        'sample': text[:700],
    }

## 2. LangGraph: состояние и узлы агента

In [ ]:
class LabAgentState(TypedDict, total=False):
    language: str
    commons_category: str
    commons_files: List[Dict[str, Any]]
    selected_file: Dict[str, Any]
    image_path: str
    ocr_text: str
    ocr_diagnostics: Dict[str, Any]
    wiki_pages: List[Dict[str, Any]]
    wiki_stats: Dict[str, Any]
    report: Dict[str, Any]
    human_tasks: List[str]

def source_scout_node(state: LabAgentState):
    files = commons_category_files(state['commons_category'])
    if not files:
        files = [{'title': FALLBACK_FILE_TITLE, 'pageid': None}]
    df = pd.DataFrame(files)
    save_artifact('lesson01_commons_candidates.csv', df)
    return {'commons_files': files}

def metadata_node(state: LabAgentState):
    inspected = []
    selected = None
    for item in state['commons_files']:
        info = commons_imageinfo(item['title'])
        inspected.append(info)
        if info.get('url') and (info.get('mime') or '').startswith('image/') and selected is None:
            selected = info
    if selected is None:
        selected = commons_imageinfo(FALLBACK_FILE_TITLE)
    save_artifact('lesson01_commons_sources.csv', pd.DataFrame(inspected))
    return {'selected_file': selected}

def ocr_node(state: LabAgentState):
    selected = state['selected_file']
    image_url = selected['url']
    suffix = Path(image_url.split('?')[0]).suffix or '.jpg'
    image_path = DATA_DIR / ('lesson01_ocr_source' + suffix)
    raw = requests.get(image_url, timeout=60, headers={'User-Agent': 'lowres-course-colab/1.0'}).content
    image_path.write_bytes(raw)

    img = Image.open(image_path).convert('RGB')
    if max(img.size) > 2200:
        img.thumbnail((2200, 2200))
        resized_path = DATA_DIR / 'lesson01_ocr_source_resized.jpg'
        img.save(resized_path)
        image_path = resized_path

    text = pytesseract.image_to_string(Image.open(image_path), lang='rus')
    save_artifact('lesson01_ocr_raw.txt', text)
    return {'image_path': str(image_path), 'ocr_text': text}

def ocr_diagnostics_node(state: LabAgentState):
    diag = text_diagnostics(state.get('ocr_text', ''))
    tasks = []
    if diag['chars'] < 100:
        tasks.append('OCR почти ничего не извлек: выбрать другой скан или улучшить предобработку.')
    if diag['cyrillic_share'] < 0.7:
        tasks.append('В тексте много некириллического шума: проверить язык OCR и качество изображения.')
    if diag['short_token_share'] > 0.45:
        tasks.append('Много коротких фрагментов: нужна ручная проверка строк и сегментации.')
    if not tasks:
        tasks.append('Выбрать 20-30 строк и вручную оценить ошибки OCR.')
    return {'ocr_diagnostics': diag, 'human_tasks': tasks}

def wiki_probe_node(state: LabAgentState):
    pages = fetch_udm_wiki_pages(limit=5)
    save_artifact('lesson01_udm_wiki_probe.csv', pd.DataFrame(pages))
    all_text = ' '.join(p['extract_preview'] for p in pages)
    stats = text_diagnostics(all_text)
    return {'wiki_pages': pages, 'wiki_stats': stats}

def report_node(state: LabAgentState):
    report = {
        'language': state['language'],
        'agent_type': 'tool-using workflow agent on LangGraph',
        'real_sources': {
            'commons_category': state['commons_category'],
            'selected_file_title': state['selected_file'].get('title'),
            'selected_file_url': state['selected_file'].get('url'),
            'license': state['selected_file'].get('license'),
            'wiki_pages': [p['url'] for p in state['wiki_pages']],
        },
        'ocr_diagnostics': state['ocr_diagnostics'],
        'wiki_probe_stats': state['wiki_stats'],
        'next_human_tasks': state['human_tasks'],
        'state_fields_used': [
            'language',
            'commons_category',
            'commons_files',
            'selected_file',
            'image_path',
            'ocr_text',
            'ocr_diagnostics',
            'wiki_pages',
            'wiki_stats',
            'human_tasks',
        ],
    }
    save_artifact('lesson01_real_agent_report.json', json.dumps(report, ensure_ascii=False, indent=2))
    return {'report': report}

workflow = StateGraph(LabAgentState)
workflow.add_node('source_scout', source_scout_node)
workflow.add_node('metadata', metadata_node)
workflow.add_node('ocr', ocr_node)
workflow.add_node('ocr_diagnostics', ocr_diagnostics_node)
workflow.add_node('wiki_probe', wiki_probe_node)
workflow.add_node('report', report_node)

workflow.set_entry_point('source_scout')
workflow.add_edge('source_scout', 'metadata')
workflow.add_edge('metadata', 'ocr')
workflow.add_edge('ocr', 'ocr_diagnostics')
workflow.add_edge('ocr_diagnostics', 'wiki_probe')
workflow.add_edge('wiki_probe', 'report')
workflow.add_edge('report', END)

agent = workflow.compile()

## 3. Запускаем агента

In [ ]:
state = agent.invoke({
    'language': LANGUAGE,
    'commons_category': COMMONS_CATEGORY,
})

print(json.dumps(state['report'], ensure_ascii=False, indent=2))

## 4. Смотрим реальные артефакты

In [ ]:
print('Выбранный файл:', state['selected_file']['title'])
print('Лицензия:', state['selected_file'].get('license'))
print('URL:', state['selected_file'].get('url'))

img = Image.open(state['image_path'])
print('Размер изображения для OCR:', img.size)
display(img)

In [ ]:
print(state['ocr_text'][:2000])

In [ ]:
display(pd.DataFrame(state['wiki_pages'])[['title', 'chars', 'tokens', 'cyrillic_share', 'url']])
display(pd.DataFrame([state['ocr_diagnostics'], state['wiki_stats']], index=['ocr', 'wiki_probe']))

## 5. Что здесь агентского

В этой тетрадке важны не “виды агентов”, а конкретная схема работы:

- `state` хранит текущую картину задачи: цель, найденные источники, выбранный файл, OCR-текст, диагностику, контрольный корпус и задачи для человека.
- каждый узел делает один проверяемый шаг;
- следующий шаг выбирается на основании уже собранного состояния;
- человек остается в контуре там, где нельзя автоматически решать про права, качество и публикацию.

Такой агент можно переделать под другие задачи: поиск корпуса, первичную разметку, проверку словаря, подготовку data card, маршрутизацию материалов к эксперту.

## Вопросы для отчета

1. Какой реальный материал нашел агент и можно ли понять его лицензию?
2. Получился ли OCR-текст вменяемым? Покажите 3-5 характерных ошибок.
3. Какие поля `state` реально повлияли на следующие шаги?
4. Что в этой задаче нельзя отдавать агенту полностью автоматически?
5. Как бы вы изменили этот workflow для своего языка или корпуса?